# Building multilayer networks with MuxVizPy

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import scipy.sparse as sps
import matplotlib.pyplot as plt
import graph_tool as gt
import graph_tool.draw as gtdraw

from MuxVizPy.utils import parsing, io
from MuxVizPy import versatility, global_descriptors, topology, visualization

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110

## Concepts and notation

A physical node is one entity. Its copy in one layer is a node-layer replica.
Intra-layer edges are observed links. Inter-layer edges connect replicas and form the
coupling. MuxVizPy flattens a replica as `layer * N + node` in supra matrices.

### Running example

The office network has 10 people in email, chat, and meetings. Meetings has three
separate triangles, and Jo has no meeting edges.

In [ ]:
NODE_NAMES = ["Ada", "Ben", "Cleo", "Dan", "Eve", "Femi", "Gil", "Hana", "Ivan", "Jo"]
LAYER_NAMES = ["email", "chat", "meetings"]

N, L = len(NODE_NAMES), len(LAYER_NAMES)
idx = {name: i for i, name in enumerate(NODE_NAMES)}

INTRA = {
    "email": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ada", "Dan"), ("Ben", "Cleo"),
        ("Cleo", "Dan"), ("Dan", "Eve"), ("Eve", "Femi"), ("Femi", "Gil"),
        ("Gil", "Hana"), ("Hana", "Ivan"), ("Ivan", "Jo"), ("Jo", "Ada"),
    ],
    "chat": [
        ("Ada", "Ben"), ("Ben", "Eve"), ("Eve", "Hana"), ("Hana", "Jo"),
        ("Jo", "Cleo"), ("Cleo", "Femi"), ("Femi", "Ivan"), ("Ivan", "Dan"),
        ("Dan", "Gil"), ("Gil", "Ada"), ("Ben", "Hana"),
    ],
    "meetings": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ben", "Cleo"),
        ("Dan", "Eve"), ("Dan", "Femi"), ("Eve", "Femi"),
        ("Gil", "Hana"), ("Gil", "Ivan"), ("Hana", "Ivan"),
    ],
}

print(f"{N} nodes x {L} layers = {N * L} node-layer replicas")

### Extended edge list

MuxVizPy reads `node.from`, `layer.from`, `node.to`, `layer.to`, and `weight`.
Indices start at 0 and have no gaps. Undirected edges need one row in each direction.

In [ ]:
rows = []
for layer_name, pairs in INTRA.items():
    layer = LAYER_NAMES.index(layer_name)
    for a, b in pairs:
        rows.append((idx[a], layer, idx[b], layer, 1.0))
        rows.append((idx[b], layer, idx[a], layer, 1.0))   # undirected -> both directions

edges = pl.DataFrame(
    rows,
    schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
    orient="row",
)

print(f"{edges.height} directed rows")
edges.head()

## Network forms

The sparse tensor has shape `(N, L, N, L)`. MuxVizPy converts it to a supra matrix for
centralities, a graph list for topology and plotting, or an aggregate network with the
layers combined.

In [ ]:
t = parsing.build_tensor_from_dataframe(edges)
print("tensor shape:", tuple(t.shape))
print("non-zero entries:", t._nnz())

### Binary and weighted supra matrices

`build_supra_adjacency_matrix_from_tensor` makes a binary matrix.
`build_supra_interaction_matrix_from_tensor` keeps the weights.

In [ ]:
supra_ec = parsing.build_supra_adjacency_matrix_from_tensor(t)      # binary
inter_ec = parsing.build_supra_interaction_matrix_from_tensor(t)    # weighted

print("supra-adjacency :", supra_ec.shape, "nnz =", supra_ec.nnz)
print("supra-interaction:", inter_ec.shape, "nnz =", inter_ec.nnz)

Without coupling, the supra matrix has one separate diagonal block per layer.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.spy(supra_ec, markersize=3)
for k in range(1, L):
    ax.axhline(k * N - 0.5, color="crimson", lw=0.8)
    ax.axvline(k * N - 0.5, color="crimson", lw=0.8)
ax.set_title("Edge-coloured supra-adjacency\n(block-diagonal: no coupling yet)")
ax.set_xticks([N * k + N / 2 for k in range(L)], LAYER_NAMES)
ax.set_yticks([N * k + N / 2 for k in range(L)], LAYER_NAMES)
plt.tight_layout()
plt.show()

In [ ]:
# Messages per week on the busiest ties; every other tie counts as 1.
MESSAGES = {
    ("Jo", "Ada"): 14.0, ("Ivan", "Jo"): 12.0,
    ("Hana", "Jo"): 11.0, ("Jo", "Cleo"): 9.0, ("Ada", "Ben"): 6.0,
}

rows_weighted = []
for layer_name, pairs in INTRA.items():
    layer = LAYER_NAMES.index(layer_name)
    for a, b in pairs:
        w = MESSAGES.get((a, b), MESSAGES.get((b, a), 1.0))
        rows_weighted.append((idx[a], layer, idx[b], layer, w))
        rows_weighted.append((idx[b], layer, idx[a], layer, w))

t_weighted = parsing.build_tensor_from_dataframe(
    pl.DataFrame(rows_weighted, schema=edges.columns, orient="row")
)
adj_w = parsing.build_supra_adjacency_matrix_from_tensor(t_weighted)
int_w = parsing.build_supra_interaction_matrix_from_tensor(t_weighted)

print("same entries in both:", adj_w.nnz == int_w.nnz, f"({adj_w.nnz})")
print("supra-adjacency  holds:", sorted(set(adj_w.data.tolist())))
print("supra-interaction holds:", sorted(set(int_w.data.tolist())))

### Graph lists and aggregate networks

`build_list_of_graphs_from_tensor` keeps all `N` nodes in every layer, including
isolated replicas. `build_aggregate_network_from_tensor` combines the layers with
`kind="sum"`, `"max"`, or `"min"`.

In [ ]:
g_list = parsing.build_list_of_graphs_from_tensor(t)
for name, g in zip(LAYER_NAMES, g_list):
    print(f"{name:9s} {g.num_vertices():3d} nodes  {g.num_edges():3d} directed edges")

# Every layer keeps all N vertices, even where a node has no edges.
meetings = g_list[LAYER_NAMES.index("meetings")]
assert all(g.num_vertices() == N for g in g_list)
print(f'\nJo has degree {meetings.get_total_degrees([idx["Jo"]])[0]} in meetings '
      f'but is still vertex {idx["Jo"]} of {meetings.num_vertices()}')

agg = parsing.build_aggregate_network_from_tensor(t, kind="sum")
g_agg = parsing.get_aggregate_network(g_list, obj_type="glist")
print("\naggregate:", agg.shape, "nnz =", agg.nnz,
      "| as graph:", g_agg.num_vertices(), "nodes,", g_agg.num_edges(), "edges")

### Conversion checks

The reverse conversions recover the stored entries and values.

In [ ]:
t_back = parsing.build_tensor_from_supra_adjacency_matrix(
    supra_ec, nodes=N, layers=L
)
supra_again = parsing.build_supra_adjacency_matrix_from_tensor(t_back)
print("supra -> tensor -> supra, identical matrices:", (supra_again != supra_ec).nnz == 0)

el_back = parsing.build_edgelist_from_tensor(t)
replica_cols = ["node.from", "layer.from", "node.to", "layer.to"]
print("tensor -> edge list, identical rows:",
      set(edges.select(replica_cols).rows()) == set(el_back.select(replica_cols).rows()),
      f"({el_back.height} rows, from the {edges.height} we started with)")

## Add inter-layer coupling

`build_interlayer_coupling_matrix` supports `categorical` links between every layer,
`ordered` links between neighbours, and forward `temporal` links. Zero coupling gives
an edge-coloured network.

In [ ]:
for kind in ["categorical", "ordered", "temporal"]:
    C = parsing.build_interlayer_coupling_matrix(L, omega=1.0, kind=kind)
    print(f"{kind}:")
    print(C.toarray().astype(int), "\n")

`build_supra_adjacency_matrix_from_edge_colored_matrices` adds the expanded coupling
to the observed layer blocks.

In [ ]:
node_tensor = parsing.get_node_tensor_from_network_list(g_list)   # one CSR per layer

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
supras = {"edge-coloured (none)": supra_ec}
for kind in ["categorical", "ordered", "temporal"]:
    C = parsing.build_interlayer_coupling_matrix(L, omega=1.0, kind=kind)
    supras[kind] = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(
        node_tensor, C, N
    )

for ax, (name, S) in zip(axes, supras.items()):
    ax.spy(S, markersize=2.5)
    for k in range(1, L):
        ax.axhline(k * N - 0.5, color="crimson", lw=0.7)
        ax.axvline(k * N - 0.5, color="crimson", lw=0.7)
    ax.set_title(f"{name}\nnnz = {S.nnz}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

Subsequent measures use categorical coupling.

In [ ]:
C_cat = parsing.build_interlayer_coupling_matrix(L, omega=1.0, kind="categorical")
supra = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(node_tensor, C_cat, N)

print("multiplex supra-adjacency:", supra.shape, "nnz =", supra.nnz)
print(f"of which coupling edges: {supra.nnz - supra_ec.nnz}"
      f"  (= N x L x (L-1) = {N * L * (L - 1)})")

### Read and write files

The `io` module reads and writes the extended edge-list CSV format.

In [ ]:
import tempfile, pathlib

tmp = pathlib.Path(tempfile.mkdtemp())
io.write_edgelist_from_tensor(t, tmp / "office.csv")
print((tmp / "office.csv").read_text().splitlines()[:3])

supra_file, n_file, l_file = io.read_edgelist_as_supraadjacencymatrix(tmp / "office.csv")
print(f"\nread back: {supra_file.shape}, N = {n_file}, L = {l_file}")

The readers infer `N` and `L` from the largest stored indices, so fully absent trailing
nodes or layers cannot be inferred.

## Degree and strength

`get_multi_degree` offers aggregate degree with `backend="muxvizpy"` and the sum of
per-layer degrees with `backend="hornet"`.

In [ ]:
# is_directed applies only to the hornet backend; 'muxvizpy' rejects it.
deg_agg = versatility.get_multi_degree(supra, nodes=N, layers=L)  # distinct neighbours
deg_sum = versatility.get_multi_degree(                            # total ties
    supra, nodes=N, layers=L, is_directed=False, backend="hornet"
)

deg = pd.DataFrame(
    {"distinct neighbours": deg_agg, "total ties": deg_sum},
    index=NODE_NAMES,
)
deg["redundancy"] = deg["total ties"] / deg["distinct neighbours"]
deg.sort_values("total ties", ascending=False)

In [ ]:
ax = deg[["distinct neighbours", "total ties"]].plot.bar(figsize=(9, 4), width=0.8)
ax.set_ylabel("degree")
ax.set_title("Two notions of multilayer degree")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Strength by layer

`compute_instrength` returns one weighted value for each node in each layer.

In [ ]:
strength = versatility.compute_instrength(supra, nodes=N, layers=L)  # (N, L)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(strength, cmap="viridis", aspect="auto")
ax.set_xticks(range(L), LAYER_NAMES)
ax.set_yticks(range(N), NODE_NAMES)
ax.set_title("Intra-layer strength per node")
for i in range(N):
    for j in range(L):
        ax.text(j, i, int(strength[i, j]), ha="center", va="center",
                color="w" if strength[i, j] < strength.max() * 0.6 else "k", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("Jo's meetings strength:", strength[idx["Jo"], LAYER_NAMES.index("meetings")])

### Combine layer values

`aggregate_metrics_over_layers` reduces the `(N, L)` table with `sum`, `mean`, `min`,
or `max`.

In [ ]:
per_node = pd.DataFrame({
    "sum over layers": versatility.aggregate_metrics_over_layers(strength, method="sum"),
    "strongest layer": versatility.aggregate_metrics_over_layers(strength, method="max"),
    "total ties (4.1)": deg["total ties"].to_numpy(),
}, index=NODE_NAMES)

print("summing the layers matches the hornet backend:",
      np.allclose(per_node["sum over layers"], per_node["total ties (4.1)"]))
per_node.astype(int)

In [ ]:
node_tensor_w = parsing.get_node_tensor_from_network_list(
    parsing.build_list_of_graphs_from_tensor(t_weighted)
)
supra_weighted = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(
    node_tensor_w, C_cat, N
)

busiest = pd.DataFrame({
    "ties": versatility.aggregate_metrics_over_layers(
        versatility.compute_instrength(supra, nodes=N, layers=L), method="sum"),
    "messages": versatility.aggregate_metrics_over_layers(
        versatility.compute_instrength(supra_weighted, nodes=N, layers=L), method="sum"),
}, index=NODE_NAMES).astype(int)
busiest["per tie"] = (busiest["messages"] / busiest["ties"]).round(1)

print("fewest ties:  ", busiest["ties"].idxmin())
print("most messages:", busiest["messages"].idxmax())
busiest.sort_values("messages", ascending=False)

## Spectral centralities

The four scores are eigenvector, Katz, PageRank, and classical random-walk centrality.
MuxVizPy defaults to `alpha ≈ 1 / rho` for muxViz compatibility, which makes normalised
Katz close to eigenvector centrality. `rho` is the spectral radius. The value `0.5 / rho`
gives a clearer difference.

In [ ]:
rho, leading = versatility.get_largest_eigenvalue(supra)
katz_alpha = 0.5 / rho

print(f"largest eigenvalue rho(A)  = {rho:.4f}")
print(f"Katz attenuation alpha     = {katz_alpha:.4f}   = 0.5 / rho(A)")
print(f"leading eigenvector length = {len(leading)}   = N x L, one entry per replica")

In [ ]:
cent = pd.DataFrame({
    "eigenvector": versatility.compute_eigenvector_centrality(supra, nodes=N, layers=L),
    "katz":        versatility.compute_katz_centrality(
        supra, nodes=N, layers=L, alpha=katz_alpha
    ),
    "pagerank":    versatility.compute_multipagerank_centrality(supra, nodes=N, layers=L),
    "rw":          versatility.compute_multi_rw_centrality(
        supra, nodes=N, layers=L, kind="classical"
    ),
}, index=NODE_NAMES)

print("largest gap between eigenvector and katz:",
      f'{(cent["eigenvector"] - cent["katz"]).abs().max():.4f}')
print("\nspread (max - min) of each measure:")
print((cent.max() - cent.min()).round(2))

cent.round(3).sort_values("eigenvector", ascending=False)

In [ ]:
ax = cent.plot.bar(figsize=(9, 4), width=0.82)
ax.set_ylabel("normalised centrality")
ax.set_title("Versatility measures on the categorical multiplex")
ax.legend(frameon=False, ncols=4)
plt.tight_layout()
plt.show()

### Solver choices

Katz supports the `direct`, `neumann`, `gmres`, and `bicgstab` solvers.

In [ ]:
exact = versatility.compute_katz_centrality(
    supra, nodes=N, layers=L, alpha=katz_alpha, solver="direct"
)

for solver in ("neumann", "gmres", "bicgstab"):
    approx = versatility.compute_katz_centrality(
        supra, nodes=N, layers=L, alpha=katz_alpha, solver=solver
    )
    same_order = np.array_equal(np.argsort(-approx), np.argsort(-exact))
    print(f"{solver:9} largest difference from exact {np.abs(approx - exact).max():.1e}"
          f"   same ranking: {same_order}")

In [ ]:
hub_exact = versatility.compute_multi_hub_centrality(supra, nodes=N, layers=L)
hub_approx = versatility.compute_multi_hub_centrality(
    supra, nodes=N, layers=L, approx=True
)

print(f"hub, power iteration against eigensolver: "
      f"largest difference {np.abs(hub_exact - hub_approx).max():.1e}")

### Coupling changes scores

In [ ]:
by_coupling = pd.DataFrame(
    {name: versatility.compute_eigenvector_centrality(S, nodes=N, layers=L)
     for name, S in supras.items()},
    index=NODE_NAMES,
)

ax = by_coupling.plot.bar(figsize=(9, 4), width=0.82)
ax.set_ylabel("eigenvector centrality")
ax.set_title("Same layers, four coupling choices")
ax.legend(frameon=False, ncols=4)
plt.tight_layout()
plt.show()

by_coupling.round(3)

## Directed layers

Directed edges model senders and receivers. In-strength and out-strength then differ.
HITS gives separate hub and authority scores.

In [ ]:
rows_directed = []
for layer_name, pairs in INTRA.items():
    layer = LAYER_NAMES.index(layer_name)
    for a, b in pairs:
        rows_directed.append((idx[a], layer, idx[b], layer, 1.0))   # a -> b, and not back

t_directed = parsing.build_tensor_from_dataframe(
    pl.DataFrame(rows_directed, schema=edges.columns, orient="row")
)
node_tensor_d = parsing.get_node_tensor_from_network_list(
    parsing.build_list_of_graphs_from_tensor(t_directed)
)
supra_directed = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(
    node_tensor_d, C_cat, N
)

print("undirected supra is asymmetric:", (supra != supra.T).nnz > 0)
print("directed   supra is asymmetric:", (supra_directed != supra_directed.T).nnz > 0)

flow = pd.DataFrame({
    "sent": versatility.aggregate_metrics_over_layers(
        versatility.compute_outstrength(supra_directed, nodes=N, layers=L), method="sum"),
    "received": versatility.aggregate_metrics_over_layers(
        versatility.compute_instrength(supra_directed, nodes=N, layers=L), method="sum"),
}, index=NODE_NAMES).astype(int)
flow

In [ ]:
hub_undirected = versatility.compute_multi_hub_centrality(supra, nodes=N, layers=L)
print("on undirected input, hub == eigenvector centrality:",
      np.allclose(hub_undirected, versatility.compute_eigenvector_centrality(
          supra, nodes=N, layers=L)))

roles = pd.DataFrame({
    "hub": versatility.compute_multi_hub_centrality(
        supra_directed, nodes=N, layers=L),
    "authority": versatility.compute_multi_authority_centrality(
        supra_directed, nodes=N, layers=L),
}, index=NODE_NAMES).round(3)

print("top hub:      ", roles["hub"].idxmax())
print("top authority:", roles["authority"].idxmax())
roles

In [ ]:
for name, S in supras.items():
    print(f"{name:22} asymmetric: {(S != S.T).nnz > 0}")

## Global descriptors

In [ ]:
agcc = global_descriptors.compute_average_global_clustering_coefficient(
    supra_ec, nodes=N, layers=L
)
agov = global_descriptors.compute_average_global_overlap(
    supra_ec, nodes=N, layers=L
)

print(f"average global clustering coefficient : {agcc:.4f}")
print(f"average global edge overlap           : {agov:.4f}")

Overlap and global clustering use the edge-coloured matrix.

In [ ]:
ov_edge = global_descriptors.compute_average_global_overlap_matrix(
    supra_ec, nodes=N, layers=L
)
ov_node = global_descriptors.compute_average_global_node_overlap_matrix(
    supra_ec, nodes=N, layers=L
)

print(f"edge overlap  email-meetings = {ov_edge[0, 2]:.2f}   "
      f"chat-email = {ov_edge[1, 0]:.2f}   chat-meetings = {ov_edge[1, 2]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, M, ttl in zip(axes, [ov_edge, ov_node], ["Edge overlap", "Node overlap"]):
    im = ax.imshow(M, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(L), LAYER_NAMES, rotation=30)
    ax.set_yticks(range(L), LAYER_NAMES)
    ax.set_title(ttl)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                    color="w" if M[i, j] < 0.6 else "k", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

## Connected structure

`get_multi_LCC` finds the largest component after combining layers. `get_multi_LIC`
intersects the largest component from every layer.

In [ ]:
lcc = topology.get_multi_LCC(g_list, obj_type="glist")
lic = topology.get_multi_LIC(g_list, obj_type="glist")

print(f"LCC: {len(lcc)}/{N} nodes  -> {sorted(NODE_NAMES[i] for i in lcc)}")
print(f"LIC: {len(lic)}/{N} nodes  -> {sorted(NODE_NAMES[i] for i in lic)}")

## Draw the multiplex

`plotMultiplex` draws one plane per layer. Shared positions and layer labels align the
planes.

In [ ]:
positions = gtdraw.sfdp_layout(g_agg).get_2d_array([0, 1])

visualization.plotMultiplex(
    g_list,
    g_agg,
    positions=positions,     # shared layout -> layers are comparable
    layer_labels=LAYER_NAMES,  # name each plane
    plane_alpha=0.15,          # translucent planes so edges stay readable
    size_mode="per_layer",   # node size = degree within that layer
    min_size=20.0,
    max_size=200.0,
    edge_alpha=0.35,
    edge_lw=0.8,
    elev=22,
    azim=-60,
)

## Function guide

| Task | Function |
|---|---|
| Build a tensor | `parsing.build_tensor_from_dataframe` |
| Build a supra matrix | `parsing.build_supra_*_from_tensor` |
| Build layer graphs | `parsing.build_list_of_graphs_from_tensor` |
| Combine layers | `parsing.build_aggregate_network_from_tensor` |
| Add coupling | `parsing.build_interlayer_coupling_matrix` |
| Degree and strength | `versatility.get_multi_degree`, `compute_instrength` |
| Spectral scores | `versatility.compute_*centrality` |
| Overlap and clustering | `global_descriptors.compute_average_global_*` |
| Connected parts | `topology.get_multi_LCC`, `get_multi_LIC` |
| Draw layers | `visualization.plotMultiplex` |